# Training 13 · Enhancers post-trained with a predictor as reward (E3)

The released pre-trained enhancer is post-trained with GRPO for 2500 steps using one predictor's score as reward
(R0, R3, R2 and the three baselines; 3 seeds, R2 seed 0 only), about 20 min each. Evaluation-only counterpart:
`tables/table3_downstream.ipynb`.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e3_grpo']]
print(len(KEYS), "runs:", ", ".join(KEYS))

16 runs: e3_grpo_R0_s0, e3_grpo_R3_s0, e3_grpo_baseline_noise_s0, e3_grpo_baseline_pgd_s0, e3_grpo_baseline_random_s0, e3_grpo_R0_s1, e3_grpo_R3_s1, e3_grpo_baseline_noise_s1, e3_grpo_baseline_pgd_s1, e3_grpo_baseline_random_s1, e3_grpo_R0_s2, e3_grpo_R3_s2, e3_grpo_baseline_noise_s2, e3_grpo_baseline_pgd_s2, e3_grpo_baseline_random_s2, e3_grpo_R2_s0


In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: se_grpo
task: se_grpo
phase: main
seed: 0
run_name: se_grpo_R0_s0__2026-09-15_17-32-04
init_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/se_pretrain_s0__2026-09-15_16-10-03.checkpoints.best.pt
policy:
  action_bands: 32
  init_log_std: -1.0
  min_log_std: -3.0
  max_log_std: 0.0
reward:
  ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
  pretrained: facebook/wav2vec2-base
  cross_ckpts:
    R3: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/hnp_s0.round3.predictor.last.pt
grpo:
  steps: 2500
  batch_size: 16
  group_size: 8
  crop_seconds: 3.0
  lr: 0.0001
  lr_log_std: 0.001
  learn_sigma: true
  betas:
  - 0.9
  - 0.999
  ppo_epochs: 2
  clip: 0.2
  adv_eps: 0.0001
  grad_clip: 1.0
  kl_mode: loss
  kl_coef: 0.0
  log_every: 10
  eval_every: 250
data:
  train_split: train
  num_workers: 6
  max_train_utts: null
  subset_see

In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for e3_grpo_R0_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_e3_grpo_R0_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key e3_grpo_R0_s0



## Retrained vs original

In [6]:
released = repro.fetch_results()
rows = {}
for k in KEYS:
    for src, d in (("original", released / Path(reg[k]["run_dir"]).name), ("retrained", repro.RUNS / "retrain" / k)):
        f = d / "results" / "val_step002500.csv"
        if f.exists():
            rows[(k, src)] = pd.read_csv(f).mean(numeric_only=True)
pd.DataFrame(rows).T.round(3) if rows else "no runs found"

dnsmos_bak  dnsmos_ovrl  dnsmos_p808  dnsmos_sig  energy_db  estoi  nisqa_mos   pesq  score_R0  score_R3  score_reward   sisdr  spk_sim  utmos    wer
e3_grpo_R0_s0              original        3.830        3.072        3.426       3.452      1.465  0.687      3.883  2.088       NaN     0.953         3.549  11.520    0.901  3.467  0.157
                           retrained       3.830        3.072        3.426       3.452      1.465  0.687      3.883  2.088       NaN     0.953         3.549  11.520    0.901  3.467  0.157
e3_grpo_R3_s0              original        3.908        3.071        3.442       3.408     -0.876  0.712      4.129  2.156     3.437       NaN         3.545  11.797    0.918  3.556  0.136
e3_grpo_baseline_noise_s0  original        3.850        3.061        3.499       3.435     -0.232  0.691      4.170  2.127     3.425     1.304         3.537  10.452    0.905  3.485  0.129
e3_grpo_baseline_pgd_s0    original        3.851        3.001        3.385       3.370     -0.626  0.685      4.140  2.190     3.486     1.228         3.583   9.615    0.920  3.482  0.146
e3_grpo_baseline_random_s0 original        4.036        3.077        3.451       3.344     -0.654  0.701      4.334  2.212     3.428     2.853         3.714  11.160    0.897  3.493  0.136
e3_grpo_R0_s1              original        3.777        3.033        3.468       3.438      0.592  0.683      4.157  2.188       NaN     1.629         3.650  11.200    0.896  3.435  0.124
e3_grpo_R3_s1              original        4.050        3.170        3.368       3.444     -1.203  0.703      4.017  2.223     3.477       NaN         3.382  10.259    0.914  3.457  0.141
e3_grpo_baseline_noise_s1  original        3.846        3.039        3.414       3.411      0.518  0.693      4.326  2.175     3.612     2.353         3.560  10.265    0.891  3.466  0.120
e3_grpo_baseline_pgd_s1    original        3.894        3.026        3.465       3.371      0.993  0.698      4.290  2.240     3.591     2.407         3.539  10.684    0.899  3.503  0.124
e3_grpo_baseline_random_s1 original        3.940        3.071        3.513       3.398     -0.302  0.702      4.432  2.192     3.599     2.428         3.668  11.365    0.889  3.505  0.128
e3_grpo_R0_s2              original        3.721        3.043        3.475       3.494     -0.996  0.662      4.280  1.906       NaN     2.440         3.695   9.248    0.925  3.436  0.120
e3_grpo_R3_s2              original        3.887        3.060        3.414       3.414     -0.299  0.692      4.218  2.107     3.510       NaN         3.587  11.539    0.904  3.486  0.123
e3_grpo_baseline_noise_s2  original        3.627        2.949        3.343       3.428      0.801  0.673      3.702  2.179     3.573     3.054         3.652   9.941    0.927  3.368  0.132
e3_grpo_baseline_pgd_s2    original        3.749        3.003        3.359       3.422      1.137  0.682      3.891  2.086     3.555     3.039         3.535  10.783    0.916  3.442  0.133
e3_grpo_baseline_random_s2 original        3.971        3.020        3.398       3.317      0.181  0.691      4.159  2.176     3.509     3.505         3.520  10.996    0.927  3.493  0.142
e3_grpo_R2_s0              original        3.970        3.055        3.502       3.357     -1.385  0.705      4.397  2.173     3.429     2.195         3.605  10.981    0.898  3.495  0.132